# 01 — Load Structure and Extract Residues

In this step, we load the 3D structure of hen egg-white lysozyme (1LYZ) and build a residue-level representation of the protein.

This includes:
- parsing the PDB structure
- extracting chain A
- converting residues to 1-letter sequence
- building a structured residue table

This table will be used later for mutation generation and stability analysis.

***Imports***

In [1]:
from pathlib import Path
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
import pandas as pd

***Load structure***

In [2]:
# Define path
pdb_path = Path("../data/input/1LYZ.pdb")

# Initialize parser
parser = PDBParser(QUIET=True)

# Load structure
structure = parser.get_structure("1LYZ", pdb_path)

print("Structure loaded successfully")

Structure loaded successfully


***Access model and chain***

In [3]:
model = structure[0]
chain = model["A"]

print("Model:", model)
print("Chain:", chain)

Model: <Model id=0>
Chain: <Chain id=A>


***Extract residues***

In [10]:
residue_rows = []

for residue in chain:
    resname = residue.get_resname()   # e.g. "ALA"
    resseq = residue.id[1]            # residue number

    try:
        aa = seq1(resname)            # convert to 1-letter
    except:
        continue  # skip non-standard residues

    residue_rows.append({
        "position": resseq,
        "resname_3": resname,
        "resname_1": aa
    })

print("First 10 residues:")
for r in residue_rows[:10]:
    print(r)

print("\nTotal residues:", len(residue_rows))

First 10 residues:
{'position': 1, 'resname_3': 'LYS', 'resname_1': 'K'}
{'position': 2, 'resname_3': 'VAL', 'resname_1': 'V'}
{'position': 3, 'resname_3': 'PHE', 'resname_1': 'F'}
{'position': 4, 'resname_3': 'GLY', 'resname_1': 'G'}
{'position': 5, 'resname_3': 'ARG', 'resname_1': 'R'}
{'position': 6, 'resname_3': 'CYS', 'resname_1': 'C'}
{'position': 7, 'resname_3': 'GLU', 'resname_1': 'E'}
{'position': 8, 'resname_3': 'LEU', 'resname_1': 'L'}
{'position': 9, 'resname_3': 'ALA', 'resname_1': 'A'}
{'position': 10, 'resname_3': 'ALA', 'resname_1': 'A'}

Total residues: 230


***Build sequence***

In [11]:
from Bio.PDB.Polypeptide import is_aa

clean_residues = []

for residue in chain.get_residues():
    if is_aa(residue, standard=True):
        clean_residues.append(residue)

sequence = "".join([seq1(r.get_resname()) for r in clean_residues])

print("Sequence:")
print(sequence)
print("Length:", len(sequence))

Sequence:
KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL
Length: 129


***create dataframe***

In [12]:
df = pd.DataFrame(residue_rows)

df.head()

,position,resname_3,resname_1
0,1,LYS,K
1,2,VAL,V
2,3,PHE,F
3,4,GLY,G
4,5,ARG,R


***save to csv***

In [13]:
output_path = Path("../data/processed/residue_table.csv")

df.to_csv(output_path, index=False)

print(f"Saved residue table to: {output_path}")

Saved residue table to: ../data/processed/residue_table.csv
